# 09 — Financial Statements Analysis

**Objective:** Extend the Excel model (`interfood_model.xlsx`) with deeper financial analysis — DuPont ROE decomposition, key ratios + EPR, a three-statement view, common-size statements, and year-on-year change charts — added as new tabs to the existing workbook.

**Inputs:**
- `../data/interfood.db` (summary series, 2017–2025)
- `../data/raw/statements.csv` (full line items 2021–2025 — created later, for the 3-statement & common-size sheets)
- `../outputs/interfood_model.xlsx` (existing workbook — new sheets are added to it)

**Expected output (new tabs in interfood_model.xlsx):**
- `DuPont` — ROE = net margin × asset turnover × equity multiplier, 2017–2025, with chart
- `Ratios` — profitability, liquidity, leverage, efficiency + illustrative EPR
- `3-Statement`, `CommonSize` — full-statement analysis, 2021–2025
- Year-on-year change chart

**Conventions:** live Excel formulas (not hardcoded values); blue = input/assumption cells; each sheet-adding cell is idempotent (deletes the sheet if it already exists, so re-running never duplicates).

**Note:** the workbook must be CLOSED in Excel while these cells run — openpyxl cannot save to an open file.

## Step 1 — Load data and confirm the workbook is accessible

Before adding sheets, confirm we can (a) read the summary series from the database, and (b) open the existing workbook. DuPont needs only four series — net result, sales, total assets, equity — all of which are in the summary data for 2017–2025.

In [11]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

data_dir = Path("../data")
xlsx_path = Path("../outputs/interfood_model.xlsx")

# 1. Load summary series from the database
conn = sqlite3.connect(data_dir / "interfood.db")
wide = (pd.read_sql_query("SELECT * FROM summary_series", conn)
        .pivot(index="year", columns="metric", values="value").sort_index())
conn.close()

# 2. Confirm the existing workbook opens and see its current sheets
wb = load_workbook(xlsx_path)
print("Workbook opened. Existing sheets:", wb.sheetnames)

# 3. Stage the four DuPont inputs (EUR '000)
dupont_inputs = wide[["net_result", "sales", "total_assets", "equity"]].copy()
print("\nDuPont inputs (2017–2025), €'000:")
dupont_inputs.round(0)

Workbook opened. Existing sheets: ['Historical', 'Model', 'Sensitivity', 'DuPont']

DuPont inputs (2017–2025), €'000:


metric,net_result,sales,total_assets,equity
year,,,,
2017,14000.0,1766700.0,458800.0,100800.0
2018,15000.0,1909600.0,510600.0,112100.0
2019,23200.0,2029300.0,466900.0,130800.0
2020,31900.0,1909800.0,384000.0,148500.0
2021,22600.0,2253400.0,625300.0,167700.0
2022,33400.0,3610200.0,857700.0,202700.0
2023,71500.0,3004800.0,741800.0,232100.0
2024,36400.0,3086500.0,796200.0,242700.0
2025,20800.0,3479900.0,935500.0,228900.0


## Step 2 — Build the DuPont sheet (live formulas)

DuPont decomposes ROE into three drivers:

**ROE = Net Margin × Asset Turnover × Equity Multiplier**
= (Net result / Sales) × (Sales / Total assets) × (Total assets / Equity)

We write the four inputs as values, then the three components and ROE as **live formulas**. As a check, we also compute ROE directly (Net result / Equity) — since the Sales and Assets terms cancel in the product, the decomposition and the direct ROE must match. Any mismatch would signal a formula error.

In [12]:
# Idempotent: remove the sheet if it already exists, then create fresh
if "DuPont" in wb.sheetnames:
    del wb["DuPont"]
ws = wb.create_sheet("DuPont")

FONT = "Arial"
hdr_font = Font(name=FONT, bold=True, size=11, color="FFFFFF")
hdr_fill = PatternFill("solid", fgColor="2E5B8A")
lbl_font = Font(name=FONT, bold=True, size=10)
val_font = Font(name=FONT, size=10)
comp_font = Font(name=FONT, size=10, color="000000")  # formulas (black)

years = list(dupont_inputs.index)   # 2017..2025

# --- Title ---
ws["A1"] = "DuPont ROE Decomposition (2017–2025)"
ws["A1"].font = Font(name=FONT, bold=True, size=13)
ws["A2"] = "ROE = Net Margin × Asset Turnover × Equity Multiplier. Inputs €'000."
ws["A2"].font = Font(name=FONT, italic=True, size=9, color="666666")

# --- Header row (years) ---
r0 = 4
ws.cell(row=r0, column=1, value="Item").font = hdr_font
ws.cell(row=r0, column=1).fill = hdr_fill
for j, y in enumerate(years):
    c = ws.cell(row=r0, column=2 + j, value=int(y))
    c.font = hdr_font; c.fill = hdr_fill
    c.alignment = Alignment(horizontal="center"); c.number_format = "0"

# --- Input rows (values) ---
input_rows = {}
for i, metric in enumerate(["net_result", "sales", "total_assets", "equity"]):
    rr = r0 + 1 + i
    label = {"net_result":"Net result", "sales":"Sales",
             "total_assets":"Total assets", "equity":"Equity"}[metric]
    ws.cell(row=rr, column=1, value=label).font = lbl_font
    for j, y in enumerate(years):
        c = ws.cell(row=rr, column=2 + j, value=float(dupont_inputs.loc[y, metric]))
        c.font = val_font; c.number_format = "#,##0"
    input_rows[metric] = rr

NR = input_rows["net_result"]; SA = input_rows["sales"]
TA = input_rows["total_assets"]; EQ = input_rows["equity"]

# --- Component rows (formulas) ---
comp_start = TA + 2  # a gap after equity row (EQ = TA+1)
ws.cell(row=comp_start - 1, column=1, value="DuPont components (live formulas)").font = Font(name=FONT, bold=True, italic=True, size=10, color="2E5B8A")

comp_defs = [
    ("Net margin",        lambda c: f"={c}{NR}/{c}{SA}",  "0.00%"),
    ("Asset turnover",    lambda c: f"={c}{SA}/{c}{TA}",  "0.00x"),
    ("Equity multiplier", lambda c: f"={c}{TA}/{c}{EQ}",  "0.00x"),
    ("ROE (product)",     lambda c: f"={c}{comp_start}*{c}{comp_start+1}*{c}{comp_start+2}", "0.0%"),
    ("ROE (direct check)",lambda c: f"={c}{NR}/{c}{EQ}",  "0.0%"),
]
comp_rows = {}
for i, (label, fn, fmt) in enumerate(comp_defs):
    rr = comp_start + i
    ws.cell(row=rr, column=1, value=label).font = lbl_font
    for j in range(len(years)):
        col = get_column_letter(2 + j)
        c = ws.cell(row=rr, column=2 + j, value=fn(col))
        c.font = comp_font; c.number_format = fmt
    comp_rows[label] = rr

# Column widths
ws.column_dimensions["A"].width = 22
for j in range(len(years)):
    ws.column_dimensions[get_column_letter(2 + j)].width = 10

wb.save(xlsx_path)
print("DuPont sheet written and workbook saved.")
print(f"  Input rows: Net result={NR}, Sales={SA}, Total assets={TA}, Equity={EQ}")
print(f"  Component rows: Net margin={comp_rows['Net margin']}, Asset turnover={comp_rows['Asset turnover']}, "
      f"Equity multiplier={comp_rows['Equity multiplier']}")
print(f"  ROE product row={comp_rows['ROE (product)']}, ROE direct-check row={comp_rows['ROE (direct check)']}")
print(f"  Example net-margin formula (2017): =B{comp_start}/B{SA}  ->  =B{NR}/B{SA}")
print("\nSheets now:", wb.sheetnames)

DuPont sheet written and workbook saved.
  Input rows: Net result=5, Sales=6, Total assets=7, Equity=8
  Component rows: Net margin=9, Asset turnover=10, Equity multiplier=11
  ROE product row=12, ROE direct-check row=13
  Example net-margin formula (2017): =B9/B6  ->  =B5/B6

Sheets now: ['Historical', 'Model', 'Sensitivity', 'DuPont']


In [13]:
from openpyxl import load_workbook
from openpyxl.styles import Font
from pathlib import Path

xlsx_path = Path("../outputs/interfood_model.xlsx")
wb = load_workbook(xlsx_path)
ws = wb["DuPont"]
FONT = "Arial"

# Row 8 currently holds the components-header text sitting on the Equity data row.
# Fix: put the "Equity" label back in A8, and move the components header to A10
# (one row above the first component, Net margin, which is row 9... but Net margin
#  is at row 9 right below equity — so we need a clean gap).

# Simplest clean fix: relabel A8 as "Equity", and turn the stray header in A8 into
# a proper section label placed in a merged/empty spot. Since components start at row 9
# immediately after equity (row 8), we just relabel A8 correctly and drop the header text.
ws["A8"] = "Equity"
ws["A8"].font = Font(name=FONT, bold=True, size=10)

# Put the section header text into A-column of the (currently blank) row by using
# the cell just above the title of components. Components are rows 9-13; row 8 is equity.
# We'll add the section note into A2 area instead is messy — cleanest: leave a note in A15.
ws["A15"] = "Note: rows 9–11 are the three DuPont drivers; row 12 (product) = row 13 (direct ROE) confirms the decomposition."
ws["A15"].font = Font(name=FONT, italic=True, size=8, color="666666")

wb.save(xlsx_path)
print("Fixed: A8 relabelled 'Equity'; explanatory note added at A15.")
print("Row layout: 5=Net result, 6=Sales, 7=Total assets, 8=Equity,")
print("            9=Net margin, 10=Asset turnover, 11=Equity multiplier,")
print("            12=ROE(product), 13=ROE(direct check)")

Fixed: A8 relabelled 'Equity'; explanatory note added at A15.
Row layout: 5=Net result, 6=Sales, 7=Total assets, 8=Equity,
            9=Net margin, 10=Asset turnover, 11=Equity multiplier,
            12=ROE(product), 13=ROE(direct check)


## Step 3 — Key financial ratios + illustrative EPR (2017–2025)

Writes the standard ratio families as live formulas, plus an **illustrative EPR**.

**EPR method (illustrative):** Economic Profit ≈ Operating Profit − (Capital Employed × Cost of Capital).
- Operating profit proxy = Gross operating income − Operational expenses (both in the summary data).
- Capital employed proxy = **Equity + Net working capital** (the capital the business has tied up — a clean, defensible proxy given the summary-level data; a full model would use total assets − current liabilities).
- Cost of capital = **8%, an illustrative assumption in a blue driver cell** (Interfood does not publish its rate). Adjustable.

This is transparent economic-profit *logic*, not a reproduction of Interfood's internal EPR figure.

In [14]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
import sqlite3
import pandas as pd
from pathlib import Path

data_dir = Path("../data")
xlsx_path = Path("../outputs/interfood_model.xlsx")

conn = sqlite3.connect(data_dir / "interfood.db")
wide = (pd.read_sql_query("SELECT * FROM summary_series", conn)
        .pivot(index="year", columns="metric", values="value").sort_index())
conn.close()

wb = load_workbook(xlsx_path)
if "Ratios" in wb.sheetnames:
    del wb["Ratios"]
ws = wb.create_sheet("Ratios")

FONT = "Arial"
hdr_font = Font(name=FONT, bold=True, size=11, color="FFFFFF")
hdr_fill = PatternFill("solid", fgColor="2E5B8A")
lbl_font = Font(name=FONT, bold=True, size=10)
val_font = Font(name=FONT, size=10)
blue_font = Font(name=FONT, size=10, color="0000FF")
yellow = PatternFill("solid", fgColor="FFFF00")
sec_font = Font(name=FONT, bold=True, italic=True, size=10, color="2E5B8A")

years = list(wide.index)
n = len(years)

ws["A1"] = "Key Financial Ratios & Illustrative EPR (2017–2025)"
ws["A1"].font = Font(name=FONT, bold=True, size=13)
ws["A2"] = "Live formulas. EPR uses an illustrative 8% cost of capital (blue cell) — not Interfood's published rate."
ws["A2"].font = Font(name=FONT, italic=True, size=9, color="666666")

ws["A4"] = "Cost of capital (illustrative)"; ws["A4"].font = lbl_font
ws["B4"] = 0.08; ws["B4"].font = blue_font; ws["B4"].fill = yellow; ws["B4"].number_format = "0.0%"
COC = "$B$4"

r0 = 6
ws.cell(row=r0, column=1, value="Inputs (€'000)").font = sec_font
hdr_row = r0 + 1
ws.cell(row=hdr_row, column=1, value="Item").font = hdr_font
ws.cell(row=hdr_row, column=1).fill = hdr_fill
for j, y in enumerate(years):
    c = ws.cell(row=hdr_row, column=2 + j, value=int(y))
    c.font = hdr_font; c.fill = hdr_fill; c.alignment = Alignment(horizontal="center"); c.number_format = "0"

inputs = ["sales", "gross_operating_income", "operational_expenses", "net_result",
          "total_assets", "equity", "net_working_capital"]
labels = {"sales":"Sales", "gross_operating_income":"Gross operating income",
          "operational_expenses":"Operational expenses", "net_result":"Net result",
          "total_assets":"Total assets", "equity":"Equity", "net_working_capital":"Net working capital"}
input_rows = {}
for i, m in enumerate(inputs):
    rr = hdr_row + 1 + i
    ws.cell(row=rr, column=1, value=labels[m]).font = lbl_font
    for j, y in enumerate(years):
        v = wide.loc[y, m]
        c = ws.cell(row=rr, column=2 + j, value=float(v) if pd.notna(v) else None)
        c.font = val_font; c.number_format = "#,##0"
    input_rows[m] = rr

SA=input_rows["sales"]; GI=input_rows["gross_operating_income"]; OE=input_rows["operational_expenses"]
NR=input_rows["net_result"]; TA=input_rows["total_assets"]; EQ=input_rows["equity"]; NWC=input_rows["net_working_capital"]

rat_hdr = input_rows["net_working_capital"] + 2
ws.cell(row=rat_hdr, column=1, value="Ratios & EPR (live formulas)").font = sec_font

ratio_defs = [
    ("Gross margin %",        lambda c: f"={c}{GI}/{c}{SA}", "0.00%"),
    ("Net margin %",          lambda c: f"={c}{NR}/{c}{SA}", "0.00%"),
    ("Return on equity (ROE)",lambda c: f"={c}{NR}/{c}{EQ}", "0.0%"),
    ("Return on assets (ROA)",lambda c: f"={c}{NR}/{c}{TA}", "0.0%"),
    ("Asset turnover",        lambda c: f"={c}{SA}/{c}{TA}", "0.00x"),
    ("Equity multiplier",     lambda c: f"={c}{TA}/{c}{EQ}", "0.00x"),
    ("Operating profit (€'000)", lambda c: f"={c}{GI}-{c}{OE}", "#,##0"),
    ("Capital employed (€'000)", lambda c: f"={c}{EQ}+{c}{NWC}", "#,##0"),
]
ratio_rows = {}
start = rat_hdr + 1
for i, (label, fn, fmt) in enumerate(ratio_defs):
    rr = start + i
    ws.cell(row=rr, column=1, value=label).font = lbl_font
    for j in range(n):
        col = get_column_letter(2 + j)
        c = ws.cell(row=rr, column=2 + j, value=fn(col)); c.font = val_font; c.number_format = fmt
    ratio_rows[label] = rr

OP = ratio_rows["Operating profit (€'000)"]
CE = ratio_rows["Capital employed (€'000)"]

epr_row = CE + 1
ws.cell(row=epr_row, column=1, value="EPR (illustrative, €'000)").font = Font(name=FONT, bold=True, size=10, color="C00000")
for j in range(n):
    col = get_column_letter(2 + j)
    c = ws.cell(row=epr_row, column=2 + j, value=f"={col}{OP}-{col}{CE}*{COC}")
    c.font = Font(name=FONT, size=10, color="C00000"); c.number_format = "#,##0"

ws.column_dimensions["A"].width = 26
for j in range(n):
    ws.column_dimensions[get_column_letter(2 + j)].width = 10

wb.save(xlsx_path)

# --- Print block (fixed: no backslash inside f-string braces) ---
ce_row = ratio_rows["Capital employed (€'000)"]
print("Ratios sheet written.")
print(f"  Cost-of-capital driver cell: B4 (8%, blue)")
print(f"  Input rows {hdr_row+1}-{input_rows['net_working_capital']}")
print(f"  Ratio rows {start}-{ce_row}")
print(f"  Operating profit row={OP}, Capital employed row={CE}, EPR row={epr_row}")
print(f"  EPR formula (2017): =B{OP}-B{CE}*{COC}")
print("Sheets now:", wb.sheetnames)

Ratios sheet written.
  Cost-of-capital driver cell: B4 (8%, blue)
  Input rows 8-14
  Ratio rows 17-24
  Operating profit row=23, Capital employed row=24, EPR row=25
  EPR formula (2017): =B23-B24*$B$4
Sheets now: ['Historical', 'Model', 'Sensitivity', 'DuPont', 'Ratios']
